<a href="https://colab.research.google.com/github/ShahadAbdullahDS/SARF-banking-nlp/blob/main/03_notebooks/03_textcnn_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SARF — TextCNN Baseline

**Run ID:** `textcnn_msa_validation_v1`  
**Seed:** 42  
**Training:** MSA train only  
**Evaluation:** MSA validation only  
**Checkpoint selection:** Highest validation Macro-F1  

## Restrictions

- Saudi test was not accessed.
- MSA test was not used.
- Synthetic corpora were not used.
- Vocabulary was built from MSA train only.
- Development pilot and official run were kept separate.

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
from pathlib import Path
import random
import json
import time
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

drive.mount("/content/drive")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
DATA_DIR = Path(
    "/content/drive/MyDrive/"
    "SARF_BANKING_NLP_PROJECT/02_processed_data"
)

RUN_DIR = Path(
    "/content/drive/MyDrive/"
    "SARF_BANKING_NLP_PROJECT/05_runs/"
    "baselines/textcnn_msa_validation_v1"
)

DEV_DIR = RUN_DIR / "dev"
MODEL_DIR = RUN_DIR / "model"

DEV_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Run directory:", RUN_DIR)
print("Development directory:", DEV_DIR)
print("Model directory:", MODEL_DIR)

print("✅ Environment ready")
print("Device:", DEVICE)
print("Data directory:", DATA_DIR)
print("Run directory:", RUN_DIR)


Mounted at /content/drive
Data directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_processed_data
Run directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/05_runs/baselines/textcnn_msa_validation_v1
Development directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/05_runs/baselines/textcnn_msa_validation_v1/dev
Model directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/05_runs/baselines/textcnn_msa_validation_v1/model
✅ Environment ready
Device: cuda
Data directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/02_processed_data
Run directory: /content/drive/MyDrive/SARF_BANKING_NLP_PROJECT/05_runs/baselines/textcnn_msa_validation_v1


In [3]:
TRAIN_PATH = DATA_DIR / "msa_train_v1.csv"
VAL_PATH = DATA_DIR / "msa_val_v1.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

# فحوص البيانات
assert {"label", "text"}.issubset(train_df.columns)
assert {"label", "text"}.issubset(val_df.columns)
assert len(train_df) == 10732
assert len(val_df) == 1229
assert train_df["label"].nunique() == 77
assert val_df["label"].nunique() == 77
assert train_df[["label", "text"]].isna().sum().sum() == 0
assert val_df[["label", "text"]].isna().sum().sum() == 0

# إنشاء mapping من train فقط وبترتيب ثابت
labels = sorted(
    train_df["label"].astype(str).unique().tolist()
)

label_to_id = {
    label: index
    for index, label in enumerate(labels)
}

id_to_label = {
    index: label
    for label, index in label_to_id.items()
}

unknown_labels = (
    set(val_df["label"].astype(str)) - set(label_to_id)
)

assert len(label_to_id) == 77
assert len(unknown_labels) == 0
assert set(label_to_id.values()) == set(range(77))

# إضافة IDs
train_df["label_id"] = (
    train_df["label"].astype(str).map(label_to_id)
)

val_df["label_id"] = (
    val_df["label"].astype(str).map(label_to_id)
)

# حفظ mapping
with open(
    MODEL_DIR / "label_mapping.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "label_to_id": label_to_id,
            "id_to_label": {
                str(key): value
                for key, value in id_to_label.items()
            }
        },
        file,
        ensure_ascii=False,
        indent=2
    )

print("✅ Data and label mapping ready")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Labels:", len(label_to_id))
print("Unknown validation labels:", len(unknown_labels))

display(train_df[["text", "label", "label_id"]].head(3))

✅ Data and label mapping ready
Train: (10732, 3)
Validation: (1229, 3)
Labels: 77
Unknown validation labels: 0


,text,label,label_id
0,ما زلت أنتظر بطاقتي؟,وصول البطاقة,76
1,ماذا أفعل إذا لم تصل بطاقتي بعد أسبوعين؟,وصول البطاقة,76
2,انا أنتظر منذ أكثر من أسبوع. هل ما زالت البطاق...,وصول البطاقة,76


In [4]:
from collections import Counter

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MAX_LENGTH = 64
MIN_FREQUENCY = 1
MAX_VOCAB_SIZE = 30000

def tokenize(text):
    """Simple, fixed word tokenizer for Arabic text."""
    text = str(text).strip()
    return re.findall(r"\w+", text, flags=re.UNICODE)

# Build vocabulary from MSA train only
token_counts = Counter()

for text in train_df["text"]:
    token_counts.update(tokenize(text))

most_common_tokens = token_counts.most_common(
    MAX_VOCAB_SIZE - 2
)

vocabulary = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for token, frequency in most_common_tokens:
    if frequency >= MIN_FREQUENCY:
        vocabulary[token] = len(vocabulary)

# Calculate sequence-length information
train_lengths = train_df["text"].map(
    lambda text: len(tokenize(text))
)

val_lengths = val_df["text"].map(
    lambda text: len(tokenize(text))
)

train_truncation_rate = float(
    (train_lengths > MAX_LENGTH).mean()
)

val_truncation_rate = float(
    (val_lengths > MAX_LENGTH).mean()
)

# Save vocabulary
with open(
    MODEL_DIR / "vocabulary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        vocabulary,
        file,
        ensure_ascii=False,
        indent=2
    )

print("✅ Vocabulary built from train only")
print("Vocabulary size:", len(vocabulary))
print("Maximum length:", MAX_LENGTH)
print(
    "Train median length:",
    float(train_lengths.median())
)
print(
    "Train maximum length:",
    int(train_lengths.max())
)
print(
    "Train truncation rate:",
    f"{train_truncation_rate:.4%}"
)
print(
    "Validation truncation rate:",
    f"{val_truncation_rate:.4%}"
)

print("\nExample tokens:")
print(tokenize(train_df.iloc[0]["text"]))

✅ Vocabulary built from train only
Vocabulary size: 5755
Maximum length: 64
Train median length: 8.0
Train maximum length: 66
Train truncation rate: 0.0186%
Validation truncation rate: 0.0000%

Example tokens:
['ما', 'زلت', 'أنتظر', 'بطاقتي']


In [5]:
class TextDataset(Dataset):
    def __init__(
        self,
        dataframe,
        vocabulary,
        max_length
    ):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label_id"].astype(int).tolist()
        self.vocabulary = vocabulary
        self.max_length = max_length

    def encode_text(self, text):
        tokens = tokenize(text)

        token_ids = [
            self.vocabulary.get(
                token,
                self.vocabulary[UNK_TOKEN]
            )
            for token in tokens
        ]

        token_ids = token_ids[:self.max_length]

        padding_length = self.max_length - len(token_ids)

        token_ids += (
            [self.vocabulary[PAD_TOKEN]] * padding_length
        )

        return torch.tensor(
            token_ids,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return {
            "input_ids": self.encode_text(
                self.texts[index]
            ),
            "label": torch.tensor(
                self.labels[index],
                dtype=torch.long
            )
        }


BATCH_SIZE = 64

train_dataset = TextDataset(
    train_df,
    vocabulary,
    MAX_LENGTH
)

val_dataset = TextDataset(
    val_df,
    vocabulary,
    MAX_LENGTH
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Test one batch
sample_batch = next(iter(train_loader))

print("✅ Dataset and DataLoaders ready")
print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print(
    "Input batch shape:",
    sample_batch["input_ids"].shape
)
print(
    "Label batch shape:",
    sample_batch["label"].shape
)
print(
    "Input dtype:",
    sample_batch["input_ids"].dtype
)

✅ Dataset and DataLoaders ready
Train examples: 10732
Validation examples: 1229
Input batch shape: torch.Size([64, 64])
Label batch shape: torch.Size([64])
Input dtype: torch.int64


In [6]:
EMBEDDING_DIM = 128
FILTER_SIZES = [3, 4, 5]
FILTERS_PER_SIZE = 100
DROPOUT = 0.5
NUM_CLASSES = 77


class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_classes,
        filter_sizes,
        filters_per_size,
        dropout,
        padding_index
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_index
        )

        self.convolutions = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=filters_per_size,
                kernel_size=filter_size
            )
            for filter_size in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            filters_per_size * len(filter_sizes),
            num_classes
        )

    def forward(self, input_ids):
        # [batch, sequence] -> [batch, sequence, embedding]
        embedded = self.embedding(input_ids)

        # Conv1d expects [batch, embedding, sequence]
        embedded = embedded.transpose(1, 2)

        pooled_outputs = []

        for convolution in self.convolutions:
            convolved = torch.relu(
                convolution(embedded)
            )

            pooled = torch.max(
                convolved,
                dim=2
            ).values

            pooled_outputs.append(pooled)

        combined = torch.cat(
            pooled_outputs,
            dim=1
        )

        combined = self.dropout(combined)

        return self.classifier(combined)


# Architecture smoke test
model_test = TextCNN(
    vocab_size=len(vocabulary),
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_CLASSES,
    filter_sizes=FILTER_SIZES,
    filters_per_size=FILTERS_PER_SIZE,
    dropout=DROPOUT,
    padding_index=vocabulary[PAD_TOKEN]
).to(DEVICE)

test_input = sample_batch["input_ids"].to(DEVICE)

with torch.no_grad():
    test_logits = model_test(test_input)

assert test_logits.shape == (
    test_input.shape[0],
    NUM_CLASSES
)

assert not torch.isnan(test_logits).any()

print("✅ TextCNN architecture passed")
print("Logits shape:", test_logits.shape)
print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model_test.parameters()
        if parameter.requires_grad
    )
)

del model_test

✅ TextCNN architecture passed
Logits shape: torch.Size([64, 77])
Trainable parameters: 913717


In [7]:
def train_one_epoch(
    model,
    data_loader,
    optimizer,
    loss_function,
    device
):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(input_ids)
        loss = loss_function(logits, labels)

        if torch.isnan(loss):
            raise ValueError("Training loss became NaN.")

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


def evaluate_model(
    model,
    data_loader,
    loss_function,
    device
):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_true_ids = []
    all_predicted_ids = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids)
            loss = loss_function(logits, labels)

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_examples += batch_size

            all_true_ids.extend(
                labels.cpu().tolist()
            )

            all_predicted_ids.extend(
                predictions.cpu().tolist()
            )

    metrics = {
        "loss": total_loss / total_examples,
        "macro_f1": f1_score(
            all_true_ids,
            all_predicted_ids,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            all_true_ids,
            all_predicted_ids,
            labels=list(range(NUM_CLASSES)),
            average="weighted",
            zero_division=0
        ),
        "accuracy": accuracy_score(
            all_true_ids,
            all_predicted_ids
        )
    }

    return {
        "metrics": metrics,
        "true_ids": all_true_ids,
        "predicted_ids": all_predicted_ids
    }


print("✅ Training and evaluation functions ready")

✅ Training and evaluation functions ready


In [8]:
# Fixed pilot subsets
pilot_train_df = pd.concat(
    [
        group.sample(
            n=min(30, len(group)),
            random_state=SEED
        )
        for _, group in train_df.groupby("label")
    ],
    ignore_index=True
)

pilot_val_df = pd.concat(
    [
        group.sample(
            n=min(5, len(group)),
            random_state=SEED
        )
        for _, group in val_df.groupby("label")
    ],
    ignore_index=True
)

pilot_train_dataset = TextDataset(
    pilot_train_df,
    vocabulary,
    MAX_LENGTH
)

pilot_val_dataset = TextDataset(
    pilot_val_df,
    vocabulary,
    MAX_LENGTH
)

pilot_generator = torch.Generator()
pilot_generator.manual_seed(SEED)

pilot_train_loader = DataLoader(
    pilot_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=pilot_generator,
    num_workers=0
)

pilot_val_loader = DataLoader(
    pilot_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# New model for pilot only
pilot_model = TextCNN(
    vocab_size=len(vocabulary),
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_CLASSES,
    filter_sizes=FILTER_SIZES,
    filters_per_size=FILTERS_PER_SIZE,
    dropout=DROPOUT,
    padding_index=vocabulary[PAD_TOKEN]
).to(DEVICE)

pilot_optimizer = torch.optim.Adam(
    pilot_model.parameters(),
    lr=1e-3
)

loss_function = nn.CrossEntropyLoss()

pilot_history = []

for epoch in range(1, 3):
    train_loss = train_one_epoch(
        pilot_model,
        pilot_train_loader,
        pilot_optimizer,
        loss_function,
        DEVICE
    )

    evaluation = evaluate_model(
        pilot_model,
        pilot_val_loader,
        loss_function,
        DEVICE
    )

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        **{
            f"val_{key}": value
            for key, value
            in evaluation["metrics"].items()
        }
    }

    pilot_history.append(epoch_result)

    print(
        f"Epoch {epoch} | "
        f"train loss={train_loss:.4f} | "
        f"val loss={evaluation['metrics']['loss']:.4f} | "
        f"macro-F1={evaluation['metrics']['macro_f1']:.4f} | "
        f"accuracy={evaluation['metrics']['accuracy']:.4f}"
    )

# Pilot checks
unique_predictions = len(
    set(evaluation["predicted_ids"])
)

assert not np.isnan(
    pilot_history[-1]["train_loss"]
)
assert unique_predictions > 1
assert len(evaluation["predicted_ids"]) == len(
    pilot_val_df
)

pd.DataFrame(pilot_history).to_csv(
    DEV_DIR / "pilot_history.csv",
    index=False
)

torch.save(
    pilot_model.state_dict(),
    DEV_DIR / "pilot_model.pt"
)

print("\n✅ Development pilot passed")
print("Pilot train rows:", len(pilot_train_df))
print("Pilot validation rows:", len(pilot_val_df))
print("Unique predicted labels:", unique_predictions)

Epoch 1 | train loss=4.3320 | val loss=3.8338 | macro-F1=0.2338 | accuracy=0.2500
Epoch 2 | train loss=3.4615 | val loss=3.2966 | macro-F1=0.3592 | accuracy=0.3776

✅ Development pilot passed
Pilot train rows: 2310
Pilot validation rows: 384
Unique predicted labels: 65


In [9]:
# Remove pilot model before official training
del pilot_model
torch.cuda.empty_cache()

# Reset seeds for the official run
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

LEARNING_RATE = 1e-3
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3

official_config = {
    "run_id": "textcnn_msa_validation_v1",
    "owner": "C",
    "model_family": "TextCNN",
    "seed": SEED,
    "train_split": "msa_train_v1",
    "validation_split": "msa_val_v1",
    "train_rows": len(train_df),
    "validation_rows": len(val_df),
    "num_labels": NUM_CLASSES,
    "vocab_size": len(vocabulary),
    "vocabulary_source": "MSA train only",
    "tokenizer": "Unicode word regex",
    "max_length": MAX_LENGTH,
    "train_truncation_rate": train_truncation_rate,
    "validation_truncation_rate": val_truncation_rate,
    "embedding_dimension": EMBEDDING_DIM,
    "filter_sizes": FILTER_SIZES,
    "filters_per_size": FILTERS_PER_SIZE,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_metric": "validation macro_f1",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_selection": "highest validation macro_f1",
    "saudi_test_accessed": False
}

with open(
    RUN_DIR / "config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        official_config,
        file,
        ensure_ascii=False,
        indent=2
    )

# Rebuild shuffled loader with a fresh generator
official_generator = torch.Generator()
official_generator.manual_seed(SEED)

official_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=official_generator,
    num_workers=0
)

official_model = TextCNN(
    vocab_size=len(vocabulary),
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_CLASSES,
    filter_sizes=FILTER_SIZES,
    filters_per_size=FILTERS_PER_SIZE,
    dropout=DROPOUT,
    padding_index=vocabulary[PAD_TOKEN]
).to(DEVICE)

optimizer = torch.optim.Adam(
    official_model.parameters(),
    lr=LEARNING_RATE
)

loss_function = nn.CrossEntropyLoss()

history = []
best_macro_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
training_start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = train_one_epoch(
        official_model,
        official_train_loader,
        optimizer,
        loss_function,
        DEVICE
    )

    evaluation = evaluate_model(
        official_model,
        val_loader,
        loss_function,
        DEVICE
    )

    val_metrics = evaluation["metrics"]

    epoch_result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
        "val_accuracy": val_metrics["accuracy"]
    }

    history.append(epoch_result)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss={train_loss:.4f} | "
        f"val loss={val_metrics['loss']:.4f} | "
        f"macro-F1={val_metrics['macro_f1']:.4f} | "
        f"weighted-F1={val_metrics['weighted_f1']:.4f} | "
        f"accuracy={val_metrics['accuracy']:.4f}"
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict":
                    official_model.state_dict(),
                "best_epoch": best_epoch,
                "best_macro_f1": best_macro_f1,
                "config": official_config
            },
            MODEL_DIR / "textcnn_best.pt"
        )

        print("  ✅ Best checkpoint saved")

    else:
        epochs_without_improvement += 1

        print(
            "  No improvement:",
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("🛑 Early stopping")
        break

training_seconds = time.time() - training_start_time

history_df = pd.DataFrame(history)

history_df.to_csv(
    RUN_DIR / "train_eval_history.csv",
    index=False
)

print("\n✅ Official training completed")
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_macro_f1, 6))
print("Runtime seconds:", round(training_seconds, 2))

Epoch 01 | train loss=3.4332 | val loss=2.2416 | macro-F1=0.5114 | weighted-F1=0.5224 | accuracy=0.5330
  ✅ Best checkpoint saved
Epoch 02 | train loss=1.9483 | val loss=1.4294 | macro-F1=0.6592 | weighted-F1=0.6667 | accuracy=0.6680
  ✅ Best checkpoint saved
Epoch 03 | train loss=1.3458 | val loss=1.1468 | macro-F1=0.6832 | weighted-F1=0.6870 | accuracy=0.6876
  ✅ Best checkpoint saved
Epoch 04 | train loss=1.0208 | val loss=0.9619 | macro-F1=0.7471 | weighted-F1=0.7443 | accuracy=0.7478
  ✅ Best checkpoint saved
Epoch 05 | train loss=0.8030 | val loss=0.8748 | macro-F1=0.7490 | weighted-F1=0.7479 | accuracy=0.7486
  ✅ Best checkpoint saved
Epoch 06 | train loss=0.6501 | val loss=0.7926 | macro-F1=0.7756 | weighted-F1=0.7763 | accuracy=0.7779
  ✅ Best checkpoint saved
Epoch 07 | train loss=0.5424 | val loss=0.7358 | macro-F1=0.7854 | weighted-F1=0.7888 | accuracy=0.7893
  ✅ Best checkpoint saved
Epoch 08 | train loss=0.4443 | val loss=0.7131 | macro-F1=0.7901 | weighted-F1=0.7923 | ac

In [10]:
# Load the best checkpoint
checkpoint_path = MODEL_DIR / "textcnn_best.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

best_model = TextCNN(
    vocab_size=len(vocabulary),
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_CLASSES,
    filter_sizes=FILTER_SIZES,
    filters_per_size=FILTERS_PER_SIZE,
    dropout=DROPOUT,
    padding_index=vocabulary[PAD_TOKEN]
).to(DEVICE)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

# Final validation using the best checkpoint
final_evaluation = evaluate_model(
    best_model,
    val_loader,
    loss_function,
    DEVICE
)

final_metrics = final_evaluation["metrics"]
true_ids = final_evaluation["true_ids"]
predicted_ids = final_evaluation["predicted_ids"]

assert len(predicted_ids) == 1229
assert len(set(predicted_ids)) > 1

# Convert label IDs back to intent names
true_labels = [
    id_to_label[label_id]
    for label_id in true_ids
]

predicted_labels = [
    id_to_label[label_id]
    for label_id in predicted_ids
]

# Save validation metrics
validation_metrics = {
    "n_examples": len(true_ids),
    "n_labels_in_protocol": NUM_CLASSES,
    "macro_f1": float(final_metrics["macro_f1"]),
    "weighted_f1": float(
        final_metrics["weighted_f1"]
    ),
    "accuracy": float(final_metrics["accuracy"]),
    "loss": float(final_metrics["loss"]),
    "best_epoch": int(checkpoint["best_epoch"]),
    "checkpoint_selection_metric":
        "highest macro_f1 on MSA validation"
}

with open(
    RUN_DIR / "validation_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        validation_metrics,
        file,
        ensure_ascii=False,
        indent=2
    )

# Save validation predictions
validation_predictions = pd.DataFrame({
    "sample_id": val_df.index,
    "text": val_df["text"],
    "y_true": true_labels,
    "y_pred": predicted_labels,
    "y_true_id": true_ids,
    "y_pred_id": predicted_ids
})

validation_predictions.to_csv(
    RUN_DIR / "validation_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save per-class metrics
report = classification_report(
    true_ids,
    predicted_ids,
    labels=list(range(NUM_CLASSES)),
    target_names=[
        id_to_label[index]
        for index in range(NUM_CLASSES)
    ],
    output_dict=True,
    zero_division=0
)

per_class_rows = []

for label_id in range(NUM_CLASSES):
    label_name = id_to_label[label_id]
    values = report[label_name]

    per_class_rows.append({
        "label_id": label_id,
        "label": label_name,
        "precision": float(values["precision"]),
        "recall": float(values["recall"]),
        "f1": float(values["f1-score"]),
        "support": int(values["support"])
    })

per_class_metrics = pd.DataFrame(per_class_rows)

per_class_metrics.to_csv(
    RUN_DIR / "per_class_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save confusion matrix
matrix = confusion_matrix(
    true_ids,
    predicted_ids,
    labels=list(range(NUM_CLASSES))
)

confusion_df = pd.DataFrame(
    matrix,
    index=[
        id_to_label[index]
        for index in range(NUM_CLASSES)
    ],
    columns=[
        id_to_label[index]
        for index in range(NUM_CLASSES)
    ]
)

confusion_df.index.name = "true_label"

confusion_df.to_csv(
    RUN_DIR / "confusion_matrix.csv",
    encoding="utf-8-sig"
)

# Update and save the official config
official_config["best_epoch"] = int(
    checkpoint["best_epoch"]
)

official_config["epochs_completed"] = len(history)
official_config["training_seconds"] = training_seconds

with open(
    RUN_DIR / "config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        official_config,
        file,
        ensure_ascii=False,
        indent=2
    )

# Reload vocabulary and label mapping
with open(
    MODEL_DIR / "vocabulary.json",
    encoding="utf-8"
) as file:
    reloaded_vocabulary = json.load(file)

with open(
    MODEL_DIR / "label_mapping.json",
    encoding="utf-8"
) as file:
    reloaded_mapping = json.load(file)

# Reload the saved checkpoint
reloaded_checkpoint = torch.load(
    MODEL_DIR / "textcnn_best.pt",
    map_location=DEVICE,
    weights_only=False
)

reloaded_model = TextCNN(
    vocab_size=len(reloaded_vocabulary),
    embedding_dim=EMBEDDING_DIM,
    num_classes=NUM_CLASSES,
    filter_sizes=FILTER_SIZES,
    filters_per_size=FILTERS_PER_SIZE,
    dropout=DROPOUT,
    padding_index=reloaded_vocabulary[PAD_TOKEN]
).to(DEVICE)

reloaded_model.load_state_dict(
    reloaded_checkpoint["model_state_dict"]
)

reloaded_model.eval()

# Reload test on three validation examples
reload_df = val_df.iloc[:3].copy()

reload_dataset = TextDataset(
    reload_df,
    reloaded_vocabulary,
    MAX_LENGTH
)

reload_loader = DataLoader(
    reload_dataset,
    batch_size=3,
    shuffle=False
)

reload_batch = next(iter(reload_loader))

with torch.no_grad():
    reload_logits = reloaded_model(
        reload_batch["input_ids"].to(DEVICE)
    )

reload_predicted_ids = torch.argmax(
    reload_logits,
    dim=1
).cpu().tolist()

reload_results = pd.DataFrame({
    "text": reload_df["text"].tolist(),
    "true_label": reload_df["label"].tolist(),
    "predicted_label": [
        id_to_label[index]
        for index in reload_predicted_ids
    ]
})

reload_results.to_csv(
    RUN_DIR / "reload_test.csv",
    index=False,
    encoding="utf-8-sig"
)

assert len(reload_results) == 3

# Save the unified evaluation summary
evaluation_summary = {
    "run_id": "textcnn_msa_validation_v1",
    "owner": "C",
    "model_family": "TextCNN",
    "condition": "baseline_textcnn",
    "run_type": "official_baseline",
    "seed": SEED,
    "train_split": "msa_train_v1",
    "validation_split": "msa_val_v1",
    "train_rows": 10732,
    "validation_rows": 1229,
    "num_labels": 77,
    "checkpoint_selection_metric":
        "macro_f1 on MSA validation",
    "best_epoch": int(checkpoint["best_epoch"]),
    "metrics": {
        "macro_f1":
            validation_metrics["macro_f1"],
        "weighted_f1":
            validation_metrics["weighted_f1"],
        "accuracy":
            validation_metrics["accuracy"]
    },
    "prediction_file":
        "validation_predictions.csv",
    "per_class_file":
        "per_class_metrics.csv",
    "confusion_matrix_file":
        "confusion_matrix.csv",
    "model_file":
        "model/textcnn_best.pt",
    "vocabulary_file":
        "model/vocabulary.json",
    "label_mapping_file":
        "model/label_mapping.json",
    "reload_test_passed": True,
    "saudi_test_accessed": False,
    "data_provenance": (
        "Official approved MSA train and validation "
        "splits loaded directly from the team Drive."
    ),
    "status": "completed"
}

with open(
    RUN_DIR / "evaluation_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        evaluation_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

# Save run log
run_log = f"""# TextCNN Run Log

## Data

- Train: `msa_train_v1.csv`
- Train rows: 10,732
- Validation: `msa_val_v1.csv`
- Validation rows: 1,229
- Labels: 77
- Vocabulary source: MSA train only
- Vocabulary size: {len(vocabulary)}
- Saudi test accessed: false
- Synthetic data used: false

## Configuration

- Seed: {SEED}
- Maximum length: {MAX_LENGTH}
- Embedding dimension: {EMBEDDING_DIM}
- Filter sizes: {FILTER_SIZES}
- Filters per size: {FILTERS_PER_SIZE}
- Dropout: {DROPOUT}
- Batch size: {BATCH_SIZE}
- Optimizer: Adam
- Learning rate: {LEARNING_RATE}
- Maximum epochs: {MAX_EPOCHS}
- Early-stopping patience: {EARLY_STOPPING_PATIENCE}
- Checkpoint rule: highest validation Macro-F1

## Result

- Best epoch: {checkpoint["best_epoch"]}
- Epochs completed: {len(history)}
- Macro-F1: {validation_metrics["macro_f1"]:.6f}
- Weighted F1: {validation_metrics["weighted_f1"]:.6f}
- Accuracy: {validation_metrics["accuracy"]:.6f}
- Runtime seconds: {training_seconds:.2f}
- Reload test: passed

## Development pilot

The two-epoch development pilot completed without NaN loss,
shape errors, or collapsed predictions. It used 2,310 training
rows and 384 validation rows.

## Data provenance

The official approved MSA train and validation files were loaded
directly from the team Drive.
"""

with open(
    RUN_DIR / "run_log.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(run_log)

# Final checks
required_files = [
    RUN_DIR / "config.json",
    RUN_DIR / "run_log.md",
    RUN_DIR / "train_eval_history.csv",
    RUN_DIR / "validation_metrics.json",
    RUN_DIR / "evaluation_summary.json",
    RUN_DIR / "validation_predictions.csv",
    RUN_DIR / "per_class_metrics.csv",
    RUN_DIR / "confusion_matrix.csv",
    MODEL_DIR / "textcnn_best.pt",
    MODEL_DIR / "vocabulary.json",
    MODEL_DIR / "label_mapping.json"
]

for path in required_files:
    assert path.exists(), f"Missing file: {path}"

assert len(validation_predictions) == 1229
assert len(per_class_metrics) == 77
assert matrix.shape == (77, 77)
assert evaluation_summary["status"] == "completed"
assert evaluation_summary["saudi_test_accessed"] is False

print("✅ Best checkpoint reloaded")
print("✅ All required artifacts created")
print("✅ Status: completed")
print(
    "Best epoch:",
    checkpoint["best_epoch"]
)
print(
    "Macro-F1:",
    round(validation_metrics["macro_f1"], 6)
)
print(
    "Weighted F1:",
    round(validation_metrics["weighted_f1"], 6)
)
print(
    "Accuracy:",
    round(validation_metrics["accuracy"], 6)
)

display(reload_results)

✅ Best checkpoint reloaded
✅ All required artifacts created
✅ Status: completed
Best epoch: 14
Macro-F1: 0.814559
Weighted F1: 0.816341
Accuracy: 0.815297


,text,true_label,predicted_label
0,هل يمكنك تتبع بطاقتي من أجلي؟,وصول البطاقة,وصول البطاقة
1,هل سأتمكن من تتبع البطاقة التي تم إرسالها إلي؟,وصول البطاقة,وصول البطاقة
2,بطاقتي ليست هنا بعد.,وصول البطاقة,وصول البطاقة


In [11]:
import sys
import json
import shutil
import datetime
import subprocess

# نسخ model artifacts إلى جذر مجلد التشغيل
for filename in [
    "textcnn_best.pt",
    "vocabulary.json",
    "label_mapping.json"
]:
    shutil.copy2(
        MODEL_DIR / filename,
        RUN_DIR / filename
    )

# توثيق الملفات المقروءة
access_log = {
    "logged_at": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "files_read": [
        str(TRAIN_PATH),
        str(VAL_PATH)
    ],
    "train_split": "msa_train_v1",
    "validation_split": "msa_val_v1",
    "saudi_test_accessed": False,
    "msa_test_accessed": False,
    "synthetic_data_accessed": False
}

with open(
    RUN_DIR / "input_access_log.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        access_log,
        file,
        ensure_ascii=False,
        indent=2
    )

# حفظ بيئة التشغيل
environment_text = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)

(RUN_DIR / "environment.txt").write_text(
    environment_text,
    encoding="utf-8"
)

# تحديث evaluation summary
summary_path = RUN_DIR / "evaluation_summary.json"

with open(summary_path, encoding="utf-8") as file:
    summary = json.load(file)

summary.update({
    "owner": "C",
    "model_file": "textcnn_best.pt",
    "vocabulary_file": "vocabulary.json",
    "label_mapping_file": "label_mapping.json",
    "input_access_log": "input_access_log.json",
    "environment_file": "environment.txt",
    "data_provenance": (
        "Official approved MSA train and validation "
        "splits loaded directly from the team Drive."
    ),
    "saudi_test_accessed": False,
    "status": "completed"
})

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2
    )

# فحص الملفات الثلاثة عشر
required_files = [
    "config.json",
    "run_log.md",
    "validation_metrics.json",
    "evaluation_summary.json",
    "validation_predictions.csv",
    "per_class_metrics.csv",
    "confusion_matrix.csv",
    "train_eval_history.csv",
    "vocabulary.json",
    "label_mapping.json",
    "textcnn_best.pt",
    "input_access_log.json",
    "environment.txt"
]

for filename in required_files:
    assert (
        RUN_DIR / filename
    ).exists(), f"Missing file: {filename}"

with open(
    RUN_DIR / "validation_metrics.json",
    encoding="utf-8"
) as file:
    final_metrics = json.load(file)

print("✅ All 13 required TextCNN files exist")
print("✅ Status:", summary["status"])
print("Best epoch:", final_metrics["best_epoch"])
print("Macro-F1:", round(final_metrics["macro_f1"], 6))
print(
    "Weighted F1:",
    round(final_metrics["weighted_f1"], 6)
)
print(
    "Accuracy:",
    round(final_metrics["accuracy"], 6)
)

✅ All 13 required TextCNN files exist
✅ Status: completed
Best epoch: 14
Macro-F1: 0.814559
Weighted F1: 0.816341
Accuracy: 0.815297


In [12]:
import json
import pandas as pd
import torch

with open(
    RUN_DIR / "validation_metrics.json",
    encoding="utf-8"
) as file:
    metrics_check = json.load(file)

with open(
    RUN_DIR / "evaluation_summary.json",
    encoding="utf-8"
) as file:
    summary_check = json.load(file)

history_check = pd.read_csv(
    RUN_DIR / "train_eval_history.csv"
)

checkpoint_check = torch.load(
    RUN_DIR / "textcnn_best.pt",
    map_location="cpu",
    weights_only=False
)

print(
    "Validation metrics:",
    metrics_check["macro_f1"]
)
print(
    "Evaluation summary:",
    summary_check["metrics"]["macro_f1"]
)
print(
    "Checkpoint:",
    checkpoint_check["best_macro_f1"]
)
print(
    "History maximum:",
    history_check["val_macro_f1"].max()
)
print(
    "Best epoch in metrics:",
    metrics_check["best_epoch"]
)
print(
    "Best epoch in checkpoint:",
    checkpoint_check["best_epoch"]
)
print(
    "Status:",
    summary_check["status"]
)

Validation metrics: 0.814558983104492
Evaluation summary: 0.814558983104492
Checkpoint: 0.814558983104492
History maximum: 0.814558983104492
Best epoch in metrics: 14
Best epoch in checkpoint: 14
Status: completed
